# Домашнее задание 2 (CNN)

Сделайте обучение сверточной нейронной сети на выбранном датасете:
- Создайте свой класс SimpleCNN с помощью PyTorch и обучите его
- Используйте предобученную сеть (например, ResNet).

Сравните полученные результаты и постройте визуализацию (графики, изображения и т д - см пример).

In [1]:
%pip install kagglehub --quiet

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


# Импорт библиотек и подготовка данных



In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Трансформации (для обеих моделей)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Содержимое датасета:", os.listdir(path))

train_dir = os.path.join(path, 'Training')
test_dir = os.path.join(path, 'Testing')

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset = ImageFolder(root=test_dir, transform=val_transform)

# Разделяем train на train и validation (80/20)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(test_dataset.classes)
print(f"Количество классов: {num_classes}")
print(f"Классы: {test_dataset.classes}")

Используемое устройство: cuda
Содержимое датасета: ['Training', 'Testing']
Количество классов: 4
Классы: ['glioma', 'meningioma', 'notumor', 'pituitary']


# Класс SimpleCNN

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Размер после 3 пулингов: 224 -> 112 -> 56 -> 28
        self.fc1 = nn.Linear(64 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

cnn_model = SimpleCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.001)

# Универсальная функция обучения

In [5]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    return train_losses, val_losses, train_accs, val_accs

# Обучение обеих моделей

In [ ]:
# 1. Обучаем свою CNN
print("Обучение SimpleCNN")
cnn_train_losses, cnn_val_losses, cnn_train_accs, cnn_val_accs = train_model(
    cnn_model, train_loader, val_loader, criterion, optimizer_cnn, epochs=5
)

# 2. Загружаем и обучаем ResNet18
print("\n Обучение ResNet18")
resnet_model = models.resnet18(weights='DEFAULT')

# Замораживаем все слои
for param in resnet_model.parameters():
    param.requires_grad = False

# Заменяем последний слой
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, num_classes)
resnet_model = resnet_model.to(device)

# Оптимизатор только для нового слоя
optimizer_resnet = optim.Adam(resnet_model.fc.parameters(), lr=0.001)

resnet_train_losses, resnet_val_losses, resnet_train_accs, resnet_val_accs = train_model(
    resnet_model, train_loader, val_loader, criterion, optimizer_resnet, epochs=5
)

Обучение SimpleCNN
Epoch 1/5 | Train Loss: 0.7731 Acc: 0.6982 | Val Loss: 0.4523 Acc: 0.8170
Epoch 2/5 | Train Loss: 0.4151 Acc: 0.8377 | Val Loss: 0.3160 Acc: 0.8795
Epoch 3/5 | Train Loss: 0.2951 Acc: 0.8906 | Val Loss: 0.2559 Acc: 0.9116
Epoch 4/5 | Train Loss: 0.1885 Acc: 0.9346 | Val Loss: 0.2156 Acc: 0.9196


# Визуализация и сравнение

In [ ]:
plt.figure(figsize=(14, 5))

# График потерь
plt.subplot(1, 2, 1)
plt.plot(cnn_train_losses, label='CNN Train Loss')
plt.plot(cnn_val_losses, label='CNN Val Loss')
plt.plot(resnet_train_losses, label='ResNet Train Loss', linestyle='--')
plt.plot(resnet_val_losses, label='ResNet Val Loss', linestyle='--')
plt.title('Динамика потерь (Loss)')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# График точности
plt.subplot(1, 2, 2)
plt.plot(cnn_val_accs, label='CNN Val Accuracy')
plt.plot(resnet_val_accs, label='ResNet Val Accuracy')
plt.title('Точность на валидации (Accuracy)')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

# График 1: Потери (Loss)
plt.subplot(1, 2, 1)
plt.plot(cnn_train_losses, label='CNN Train Loss', color='blue')
plt.plot(cnn_val_losses, label='CNN Val Loss', color='blue', linestyle='--')
plt.plot(resnet_train_losses, label='ResNet Train Loss', color='orange')
plt.plot(resnet_val_losses, label='ResNet Val Loss', color='orange', linestyle='--')
plt.title('Динамика потерь (Loss)')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# График 2: Точность (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(cnn_val_accs, label='CNN Val Accuracy', color='blue')
plt.plot(resnet_val_accs, label='ResNet Val Accuracy', color='orange')
plt.title('Точность на валидации (Accuracy)')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Сравнение результатов обучения моделей

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import numpy as np
import torch

# 1. Создаем test_loader
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
# 2. Функция для получения предсказаний
def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 3. Получаем предсказания на тестовой выборке
y_true, y_pred_cnn = get_predictions(cnn_model, test_loader)
_, y_pred_resnet = get_predictions(resnet_model, test_loader)

# 4. Строим матрицы ошибок
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_cnn = confusion_matrix(y_true, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
axes[0].set_title('Матрица ошибок: SimpleCNN')
axes[0].set_xlabel('Предсказано')
axes[0].set_ylabel('Истина')

cm_resnet = confusion_matrix(y_true, y_pred_resnet)
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
axes[1].set_title('Матрица ошибок: ResNet18')
axes[1].set_xlabel('Предсказано')
axes[1].set_ylabel('Истина')

plt.tight_layout()
plt.show()

# 5. Выводим отчеты
print("Классификационный отчет: SimpleCNN")
print(classification_report(y_true, y_pred_cnn, target_names=test_dataset.classes))

print("Классификационный отчет: ResNet18")
print(classification_report(y_true, y_pred_resnet, target_names=test_dataset.classes))

# Вывод


В ходе работы я сравнила свою SimpleCNN с предобученной ResNet18. В моем эксперименте простая сеть показала даже более высокую точность (около 94% против 89% у ResNet).

Я думаю,
дело в том, что для ResNet я использовала замороженные веса (feature extractor) и обучала только последний слой всего 5 эпох. Предобученные на ImageNet признаки (текстуры, контуры) не всегда идеально подходят для МРТ-снимков без дополнительного дообучения (fine-tuning). Простая CNN училась с нуля прямо на этих данных и успела лучше подстроиться под их специфику.

Анализ матриц ошибок показал, что обе модели хорошо распознают отсутствие опухоли (notumor). Основные сложности возникают при различении глиомы и менингиомы — эти классы похожи, и обе сети их иногда путают. Также важно отметить, что с медицинской точки зрения самая опасная ошибка — это пропустить опухоль (ложноотрицательное срабатывание). В дальнейшем для улучшения ResNet стоило бы разморозить несколько последних слоев и увеличить время обучения, чтобы раскрыть её потенциал.


# Новый раздел